Step 1: Load Dataset

In [1]:
import pandas as pd

df_train = pd.read_csv('/content/train.csv')

print(df_train.shape)
print(df_train.info())
print(df_train.isnull().sum())
print(df_train.duplicated().sum())

print(df_train.head())

(159571, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB
None
id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64
0
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background

In [2]:
df_test = pd.read_csv('/content/test.csv')

print(df_test.shape)
print(df_test.info())
print(df_test.isnull().sum())
print(df_test.duplicated().sum())

print(df_test.head())

(153164, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            153164 non-null  object
 1   comment_text  153164 non-null  object
dtypes: object(2)
memory usage: 2.3+ MB
None
id              0
comment_text    0
dtype: int64
0
                 id                                       comment_text
0  00001cee341fdb12  Yo bitch Ja Rule is more succesful then you'll...
1  0000247867823ef7  == From RfC == \n\n The title is fine as it is...
2  00013b17ad220c46  " \n\n == Sources == \n\n * Zawe Ashton on Lap...
3  00017563c3f7919a  :If you have a look back at the source, the in...
4  00017695ad8997eb          I don't anonymously edit articles at all.


In [3]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Test Processing

In [4]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') # Keep if specifically needed, but often 'punkt' is sufficient for tokenization

# Reload df to ensure 'comment_text' is string data
df = pd.read_csv('/content/train.csv')

# Initialize Objects
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# text cleaning

def clean_text(text):
  text = str(text).lower()
  # Replace newlines with space
  text = re.sub(r'\n', ' ', text)
  text = re.sub(r"http\S+|www\S+|https\S+","",text,flags = re.MULTILINE)
  text = re.sub(r"\@\w+|\#","",text)
  text = re.sub(r"[^a-zA-Z0-9]"," ",text)
  # Normalize whitespace: replace multiple spaces with a single space and strip leading/trailing spaces
  text = re.sub(r'\s+', ' ', text).strip()

  # tokenization
  words = word_tokenize(text)

  # stopword removal + lemmatization
  words = [
      lemmatizer.lemmatize(word)
      for word in words
      if word not in stop_words
  ]

  return " ".join(words)

df["comment_text"] = df["comment_text"].apply(clean_text)

df.head()

# TF-IDF Vectorization

tfidf = TfidfVectorizer(
    max_features=5000
)

X_tfidf = tfidf.fit_transform(
    df['comment_text']
)

print(
"TFIDF Shape:",
X_tfidf.shape
)

# Deep Learning Tokenizer

max_words = 10000
max_len = 100

tokenizer = Tokenizer(
    num_words=max_words
)

tokenizer.fit_on_texts(
    df['comment_text']
)

sequences = tokenizer.texts_to_sequences(
    df['comment_text']
)

X_pad = pad_sequences(
    sequences,
    maxlen=max_len
)

print(
"Padded Shape:",
X_pad.shape
)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


TFIDF Shape: (159571, 5000)
Padded Shape: (159571, 100)


In [5]:
df.to_csv("Train_processed_text.csv", index=False)
print("File is downloaded with all columns.")

File is downloaded with all columns.


In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') # Keep if specifically needed, but often 'punkt' is sufficient for tokenization

# Reload df to ensure 'comment_text' is string data
df = pd.read_csv('/content/test.csv')

# Initialize Objects
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# text cleaning

def clean_text(text):
  text = str(text).lower()
  # Replace newlines with space
  text = re.sub(r'\n', ' ', text)
  text = re.sub(r"http\S+|www\S+|https\S+","",text,flags = re.MULTILINE)
  text = re.sub(r"\@\w+|\#","",text)
  text = re.sub(r"[^a-zA-Z0-9]"," ",text)
  # Normalize whitespace: replace multiple spaces with a single space and strip leading/trailing spaces
  text = re.sub(r'\s+', ' ', text).strip()

  # tokenization
  words = word_tokenize(text)

  # stopword removal + lemmatization
  words = [
      lemmatizer.lemmatize(word)
      for word in words
      if word not in stop_words
  ]

  return " ".join(words)

df["comment_text"] = df["comment_text"].apply(clean_text)

df.head()

# TF-IDF Vectorization

tfidf = TfidfVectorizer(
    max_features=5000
)

X_tfidf = tfidf.fit_transform(
    df['comment_text']
)

print(
"TFIDF Shape:",
X_tfidf.shape
)

# Deep Learning Tokenizer

max_words = 10000
max_len = 100

tokenizer = Tokenizer(
    num_words=max_words
)

tokenizer.fit_on_texts(
    df['comment_text']
)

sequences = tokenizer.texts_to_sequences(
    df['comment_text']
)

X_pad = pad_sequences(
    sequences,
    maxlen=max_len
)

print(
"Padded Shape:",
X_pad.shape
)


In [ ]:
df.to_csv("Test_processed_text.csv", index=False)
print("File is downloaded with all columns.")

Label Creation

In [6]:
toxicity_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

y = df[toxicity_columns]

In [7]:
print(y.shape)
print(y.head())

(159571, 6)
   toxic  severe_toxic  obscene  threat  insult  identity_hate
0      0             0        0       0       0              0
1      0             0        0       0       0              0
2      0             0        0       0       0              0
3      0             0        0       0       0              0
4      0             0        0       0       0              0


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}")

Using cpu


Train/Test

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y,
    test_size=0.2,
    random_state=42
)

RNN Model

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

rnn_model = Sequential()

rnn_model.add(
    Embedding(
        input_dim=10000,
        output_dim=128
        # input_length=100  # Deprecated and removed
    )
)

rnn_model.add(
    SimpleRNN(
        64
    )
)

rnn_model.add(
    Dropout(.5)
)

rnn_model.add(
    Dense(
        6, # Changed from 1 to 6 to match the 6 toxicity labels
        activation='sigmoid'
    )
)

# Stop if validation loss stops improving
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Reduce learning rate automatically
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1
)

rnn_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = rnn_model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        early_stop,
        reduce_lr
    ]
)

Epoch 1/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 75s 23ms/step - accuracy: 0.8453 - loss: 0.0936 - val_accuracy: 0.9697 - val_loss: 0.0676 - learning_rate: 0.0010
Epoch 2/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9659 - loss: 0.0720
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 82s 23ms/step - accuracy: 0.9676 - loss: 0.0694 - val_accuracy: 0.9640 - val_loss: 0.0795 - learning_rate: 0.0010
Epoch 3/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 73s 23ms/step - accuracy: 0.9609 - loss: 0.0590 - val_accuracy: 0.9772 - val_loss: 0.0599 - learning_rate: 5.0000e-04
Epoch 4/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9474 - loss: 0.0543
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 82s 23ms/step - accuracy: 0.9297 - loss: 0.0557 - val_accuracy: 0.9836 - val_loss: 0.0629 - learning_rate: 5.0000e-04
Epoch 5/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - acc

In [17]:
loss, acc = rnn_model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", acc)
print("Test Loss:", loss)

rnn_model.save(
    "rnn_model.h5"
)

# Save tokenizer

import pickle
import os

# Create the directory if it doesn't exist
os.makedirs('models_rnn', exist_ok=True)

pickle.dump(
    tokenizer,
    open(
        'models_rnn/tokenizer.pkl',
        'wb'
    )
)

print("Model and tokenizer saved")

998/998 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9787 - loss: 0.0587


Test Accuracy: 0.9786620736122131
Test Loss: 0.05870547518134117
Model and tokenizer saved


LSTM Model

In [12]:
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

lstm_model=Sequential()

lstm_model.add(
Embedding(
input_dim=10000,
output_dim=128
# input_length=100 # Deprecated and removed
)
)

lstm_model.add(
Bidirectional(
LSTM(
64
)
)
)

lstm_model.add(
Dropout(.3)
)

lstm_model.add(
Dense(
64,
activation='relu'
)
)

lstm_model.add(
Dense(
6, # Changed from 1 to 6 to match the 6 toxicity labels
activation='sigmoid'
)
)

lstm_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 259s 80ms/step - accuracy: 0.9794 - loss: 0.0660 - val_accuracy: 0.9942 - val_loss: 0.0523
Epoch 2/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 255s 80ms/step - accuracy: 0.9884 - loss: 0.0474 - val_accuracy: 0.9649 - val_loss: 0.0509
Epoch 3/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 256s 80ms/step - accuracy: 0.9676 - loss: 0.0410 - val_accuracy: 0.9920 - val_loss: 0.0512
Epoch 4/15
3192/3192 ━━━━━━━━━━━━━━━━━━━━ 261s 80ms/step - accuracy: 0.9647 - loss: 0.0361 - val_accuracy: 0.9924 - val_loss: 0.0552


In [18]:
loss, acc = lstm_model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", acc)
print("Test Loss:", loss)

lstm_model.save(
    "lstm_model.h5"
)

# Save tokenizer

import pickle
import os

# Create the directory if it doesn't exist
os.makedirs('models_Lstm', exist_ok=True)

pickle.dump(
    tokenizer,
    open(
        'models_Lstm/tokenizer.pkl',
        'wb'
    )
)

print("Model and tokenizer saved")

998/998 ━━━━━━━━━━━━━━━━━━━━ 19s 19ms/step - accuracy: 0.9691 - loss: 0.0501


Test Accuracy: 0.9691367745399475
Test Loss: 0.05014139413833618
Model and tokenizer saved


Metrics

In [14]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

pred = lstm_model.predict(X_test)

pred = (pred > 0.5).astype(int)

print(
    "F1:",
    f1_score(
        y_test,
        pred,
        average='micro'
    )
)

print(
    "Precision:",
    precision_score(
        y_test,
        pred,
        average='micro'
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        pred,
        average='micro'
    )
)

998/998 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step
F1: 0.7115463747839684
Precision: 0.851487098680323
Recall: 0.6111111111111112
